# 01. EDA del inventario de Distribuciones Sago Packaging

**Project brief, notebook 01: ingesta, calidad y exploración**

Este notebook reproduce todas las cifras del `project-brief.md` a partir del CSV descargado de
Kaggle el 13/10/2026. Sigue el orden de las lecciones del curso y usa sus mismas técnicas:

| Sección | Lección | Qué hace |
|---|---|---|
| 1. Ingesta | 02 | mirar los bytes, leer con tipos declarados, contrato de ingesta (y romperlo a propósito) |
| 2. Calidad | 03 | perfilar antes de tocar, scorecard de calidad, outliers por regla, IQR y Mahalanobis, pipeline con log |
| 3. Modelo de datos | 02 | tabla de hechos + dimensiones en Parquet, merges con `validate=`, manifiesto |
| 4. Exploración | 04 | tendencia central, dispersión, categorías, tendencias, correlación |
| 5. Hipótesis | 04 | H1 (estacionalidad), H2 (comparación de reglas), H3 (promociones) |
| 6. KPIs | 04 | valores actuales por almacén y por proveedor |
| 7. Cifras del brief | todas | comprobaciones automáticas: si los datos cambian, fallan |

El notebook vive en `project/` y lee los datos de `data/raw/supply_chain_dataset1.csv`.

## 0. Setup

In [1]:
import hashlib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from scipy.stats import chi2

IN_COLAB = "google.colab" in sys.modules
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

SEED = 42
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

# El notebook vive en project/ y los datos en data/, en la raíz del repositorio
RAW = Path("..") / "data" / "raw" / "supply_chain_dataset1.csv"
PROCESSED = Path("..") / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)  # data/ ya existe: ahí está el CSV

COMBO = ["SKU_ID", "Warehouse_ID"]  # una combinación producto-almacén
KEY = ["Date", "SKU_ID", "Warehouse_ID"]  # una fila
WINDOW, MIN_DAYS, MARGIN = 28, 7, 14  # ventana de demanda, historia mínima, días de margen

print("pandas", pd.__version__)

pandas 3.0.2


## 1. Ingesta (lección 02)

La tabla de fuentes es el mapa de riesgos del proyecto: quién es el dueño, en qué formato llega,
cada cuánto se actualiza y cuánto ocupa.

In [2]:
SOURCES = pd.DataFrame(
    [
        # source,                 origin,     format, owner,             refresh, rows_per_day
        ("supply_chain_inventory", "machine", "CSV", "Ziya07 (Kaggle)", "anual", 250),
    ],
    columns=["source", "origin", "format", "owner", "refresh", "rows_per_day"],
)
SOURCES

,source,origin,format,owner,refresh,rows_per_day
0,supply_chain_inventory,machine,CSV,Ziya07 (Kaggle),anual,250


> **Lectura.** Una sola fuente, publicada por alguien **ajeno a la empresa**: puede cambiar de
> forma o desaparecer sin aviso. Por eso se guarda la copia original y se valida cada carga.

Antes de leer, mirar los bytes: separador, decimales, formato de fecha y fin de línea no vienen
escritos en un CSV.

In [3]:
with open(RAW, "rb") as fh:
    head = fh.read(300)
print(head)

b'Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast\r\n2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52\r\n2024-01-02,SKU_1,WH_1,SUP_8,West,17,57'


Coma como separador, punto decimal, fechas `aaaa-mm-dd` y fin de línea Windows (`\r\n`), que
pandas maneja solo. Leemos declarando los tipos: los identificadores como `category` y la fecha con
su formato explícito.

In [4]:
raw = pd.read_csv(
    RAW,
    parse_dates=["Date"],
    date_format="%Y-%m-%d",
    dtype={c: "category" for c in ["SKU_ID", "Warehouse_ID", "Supplier_ID", "Region"]},
)
raw.info()
raw.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     91250 non-null  datetime64[us]
 1   SKU_ID                   91250 non-null  category      
 2   Warehouse_ID             91250 non-null  category      
 3   Supplier_ID              91250 non-null  category      
 4   Region                   91250 non-null  category      
 5   Units_Sold               91250 non-null  int64         
 6   Inventory_Level          91250 non-null  int64         
 7   Supplier_Lead_Time_Days  91250 non-null  int64         
 8   Reorder_Point            91250 non-null  int64         
 9   Order_Quantity           91250 non-null  int64         
 10  Unit_Cost                91250 non-null  float64       
 11  Unit_Price               91250 non-null  float64       
 12  Promotion_Flag           91250 non-null  in

,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.95,20.48,0,0,18.63
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.95,20.48,1,0,39.62


### 1.1 Contrato de ingesta

El `check_contract` de la lección 02, con una diferencia: aquí la clave es **compuesta** (fecha +
SKU + almacén).

In [5]:
TYPE_CHECKS = {
    "text": lambda s: pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype),
    "number": pd.api.types.is_numeric_dtype,
    "datetime": pd.api.types.is_datetime64_any_dtype,
}

CONTRACT = {
    "min_rows": 5_000,  # criterio de elegibilidad del brief
    "key": KEY,
    "required": {
        "Date": "datetime",
        "SKU_ID": "text",
        "Warehouse_ID": "text",
        "Supplier_ID": "text",
        "Units_Sold": "number",
        "Inventory_Level": "number",
        "Supplier_Lead_Time_Days": "number",
        "Reorder_Point": "number",
        "Unit_Cost": "number",
        "Promotion_Flag": "number",
    },
    "ranges": {
        "Units_Sold": (0, 1_000),
        "Inventory_Level": (0, 10_000),
        "Supplier_Lead_Time_Days": (1, 90),
        "Unit_Cost": (0, 1_000),
    },
    "domains": {"Promotion_Flag": {0, 1}, "Stockout_Flag": {0, 1}},
    "max_null_share": {c: 0.0 for c in KEY + ["Units_Sold", "Inventory_Level"]},
}


def check_contract(df, contract):
    """Devuelve la lista de incumplimientos. Una lista vacía significa que los datos pasan."""
    problems = []

    if len(df) < contract["min_rows"]:
        problems.append(f"{len(df):,} filas, por debajo del mínimo {contract['min_rows']:,}")

    missing = [c for c in contract["required"] if c not in df.columns]
    if missing:
        problems.append(f"faltan columnas: {missing}")

    for col, kind in contract["required"].items():
        if col in df.columns and not TYPE_CHECKS[kind](df[col]):
            problems.append(f"{col}: el tipo {str(df[col].dtype)!r} no es {kind}")

    key = contract["key"]
    if all(c in df.columns for c in key) and df.duplicated(subset=key).any():
        problems.append(f"{df.duplicated(subset=key).sum()} claves {key} repetidas")

    for col, (lo, hi) in contract["ranges"].items():
        if col in df.columns:
            out = ((df[col] < lo) | (df[col] > hi)).sum()
            if out:
                problems.append(f"{col}: {out} valores fuera de [{lo}, {hi}]")

    for col, allowed in contract["domains"].items():
        if col in df.columns:
            unexpected = set(df[col].dropna().unique()) - allowed
            if unexpected:
                problems.append(f"{col}: valores inesperados {sorted(unexpected)}")

    for col, limit in contract["max_null_share"].items():
        if col in df.columns:
            share = df[col].isna().mean()
            if share > limit:
                problems.append(f"{col}: {share:.1%} nulos, límite {limit:.0%}")

    return problems


violations = check_contract(raw, CONTRACT)
print("✔ datos aceptados" if not violations else "✘ contrato incumplido:")
for v in violations:
    print("  -", v)

✔ datos aceptados


Ahora lo rompemos a propósito, como en la lección 02: una validación que nadie ha visto fallar es
una validación en la que nadie puede confiar.

In [6]:
broken = pd.concat([raw, raw.head(3)], ignore_index=True)  # clave repetida
broken.loc[0, "Units_Sold"] = -5  # venta imposible
broken.loc[1, "Promotion_Flag"] = 7  # valor fuera del dominio
broken["Date"] = broken["Date"].astype(str)  # fechas llegadas como texto
broken = broken.drop(columns=["Unit_Cost"])  # columna desaparecida

for v in check_contract(broken, CONTRACT):
    print("  -", v)

  - faltan columnas: ['Unit_Cost']
  - Date: el tipo 'str' no es datetime
  - 3 claves ['Date', 'SKU_ID', 'Warehouse_ID'] repetidas
  - Units_Sold: 1 valores fuera de [0, 1000]
  - Promotion_Flag: valores inesperados [np.int64(7)]


## 2. Calidad (lección 03)

Nunca limpies una tabla que no has medido. Primero los números del *antes*, sin modificar `raw`.

In [7]:
n_rows = len(raw)
n_dup_keys = raw.duplicated(subset=KEY).sum()
missing_share = raw.isna().mean()
n_combos = len(raw[COMBO].drop_duplicates())
n_days = raw["Date"].nunique()
rows_per_day = raw.groupby("Date").agg(rows=("SKU_ID", "size"))["rows"]

profile = pd.DataFrame(
    {
        "valor": [
            n_rows,
            raw.shape[1],
            n_dup_keys,
            int((missing_share > 0).sum()),
            raw["SKU_ID"].nunique(),
            raw["Warehouse_ID"].nunique(),
            raw["Supplier_ID"].nunique(),
            raw["Region"].nunique(),
            n_combos,
            n_days,
            raw["Date"].min().strftime("%d/%m/%Y"),
            raw["Date"].max().strftime("%d/%m/%Y"),
            f"{rows_per_day.min()}-{rows_per_day.max()}",
        ]
    },
    index=[
        "filas", "columnas", "claves repetidas", "columnas con nulos", "SKU", "almacenes",
        "proveedores", "regiones", "combinaciones SKU-almacén", "días", "primer día",
        "último día", "filas por día (mín-máx)",
    ],
)
assert n_rows == n_combos * n_days, "no todas las combinaciones aparecen todos los días"
profile

,valor
filas,91250
columnas,15
claves repetidas,0
columnas con nulos,0
SKU,50
almacenes,5
proveedores,10
regiones,4
combinaciones SKU-almacén,250
días,365


¿Alguna columna no varía nunca? ¿Qué columnas son **atributos fijos** de la combinación y cuáles
cambian cada día?

In [8]:
constant = raw.columns[raw.nunique() == 1].tolist()
print("columnas constantes:", constant)

stability = (
    raw.groupby(COMBO, observed=True)[
        ["Supplier_ID", "Supplier_Lead_Time_Days", "Reorder_Point", "Unit_Cost", "Unit_Price", "Region"]
    ]
    .nunique()
    .max()
    .rename("máx. valores distintos dentro de una combinación")
)
display(stability)
print("\nreparto de Region:", raw["Region"].value_counts(normalize=True).round(3).to_dict())
print("días con pedido (Order_Quantity > 0):", f"{(raw['Order_Quantity'] > 0).mean():.1%}")

columnas constantes: ['Stockout_Flag']


Supplier_ID                1
Supplier_Lead_Time_Days    1
Reorder_Point              1
Unit_Cost                  1
Unit_Price                 1
Region                     4
Name: máx. valores distintos dentro de una combinación, dtype: int64


reparto de Region: {'North': 0.252, 'East': 0.251, 'South': 0.249, 'West': 0.248}
días con pedido (Order_Quantity > 0): 5.5%


> **Lectura.** `Stockout_Flag` vale 0 en todas las filas: no sirve ni como KPI ni como objetivo.
> Proveedor, plazo, punto de pedido y precios son **atributos fijos** de cada combinación. `Region`,
> en cambio, toma los 4 valores dentro de la misma combinación, repartidos al 25 %: no describe el
> almacén, así que las comparaciones geográficas se hacen por `Warehouse_ID`. Solo el 5,5 % de los
> días tiene un pedido.

### 2.1 ¿Fuga de información en `Demand_Forecast`?

Un pronóstico honesto se parece a las ventas, pero no **demasiado** a las del mismo día. Para
compararlo con las ventas de los días de al lado, unimos la tabla consigo misma desplazando la
fecha un día (`timedelta`), con `validate="1:1"` como en la lección 02.

In [9]:
sales = raw[KEY + ["Units_Sold"]]
day_before = sales.assign(Date=sales["Date"] + timedelta(days=1)).rename(
    columns={"Units_Sold": "ventas_dia_anterior"}
)
day_after = sales.assign(Date=sales["Date"] - timedelta(days=1)).rename(
    columns={"Units_Sold": "ventas_dia_siguiente"}
)
lagged = (
    raw[KEY + ["Units_Sold", "Demand_Forecast"]]
    .merge(day_before, on=KEY, how="left", validate="1:1")
    .merge(day_after, on=KEY, how="left", validate="1:1")
)
assert len(lagged) == len(raw), "un merge cambió el número de filas"

leak = pd.DataFrame(
    {
        "correlación con Demand_Forecast": [
            lagged["Demand_Forecast"].corr(lagged["Units_Sold"]),
            lagged["Demand_Forecast"].corr(lagged["ventas_dia_anterior"]),
            lagged["Demand_Forecast"].corr(lagged["ventas_dia_siguiente"]),
        ]
    },
    index=["ventas del mismo día", "ventas del día anterior", "ventas del día siguiente"],
)
leak

,correlación con Demand_Forecast
ventas del mismo día,0.95
ventas del día anterior,0.61
ventas del día siguiente,0.61


> **Lectura.** 0,95 con el mismo día frente a 0,61 con los días de al lado: el pronóstico parece
> hecho con las ventas que pretende predecir. No lo demuestra, pero basta para **no usarlo como
> variable** en el bloque 3 hasta descartarlo.

### 2.2 Outliers: imposibles, raros para su grupo y raros en varias dimensiones

Los tres niveles de la lección 03. Primero la **regla de dominio**: valores que no pueden ser
ciertos.

In [10]:
impossible = (
    (raw["Units_Sold"] < 0)
    | (raw["Inventory_Level"] < 0)
    | (raw["Unit_Price"] <= raw["Unit_Cost"])  # vender por debajo del coste
)
print(f"filas imposibles por regla de dominio: {impossible.sum()}")

for col in ["Units_Sold", "Inventory_Level"]:
    q1, q3 = raw[col].quantile([0.25, 0.75])
    fence = 1.5 * (q3 - q1)
    n_out = ((raw[col] < q1 - fence) | (raw[col] > q3 + fence)).sum()
    print(f"{col}: valla IQR [{q1 - fence:,.0f}, {q3 + fence:,.0f}] -> {n_out} filas fuera")

filas imposibles por regla de dominio: 0
Units_Sold: valla IQR [-8, 48] -> 110 filas fuera
Inventory_Level: valla IQR [79, 855] -> 199 filas fuera


Ninguna fila es imposible. La valla IQR sobre toda la tabla marca algunas, pero mezcla meses de
temporada alta y baja. Como en el ejercicio 6 de la lección 03, la pregunta es **¿outlier para
quién?** Comparamos cada día con su propio SKU y su propio mes.

In [11]:
fig = px.box(
    raw,
    x="Warehouse_ID",
    y="Units_Sold",
    points="outliers",
    title="Unidades vendidas por día, por almacén",
)
fig.show()

In [12]:
def flag_sales_outliers(df):
    out = df.copy()
    month = out["Date"].dt.strftime("%m")
    by_group = out.groupby([out["SKU_ID"], month], observed=True)["Units_Sold"]
    q1 = by_group.transform(lambda s: s.quantile(0.25))
    q3 = by_group.transform(lambda s: s.quantile(0.75))
    fence = 1.5 * (q3 - q1)
    out["ventas_outlier"] = (out["Units_Sold"] < q1 - fence) | (out["Units_Sold"] > q3 + fence)
    return out


flagged = flag_sales_outliers(raw)
print(f"días raros para su SKU y su mes: {flagged['ventas_outlier'].sum():,} "
      f"({flagged['ventas_outlier'].mean():.1%})")
flagged.loc[flagged["ventas_outlier"], ["Date", "SKU_ID", "Warehouse_ID", "Units_Sold"]].head()

días raros para su SKU y su mes: 1,156 (1.3%)


,Date,SKU_ID,Warehouse_ID,Units_Sold
98,2024-04-08,SKU_1,WH_1,50
101,2024-04-11,SKU_1,WH_1,49
124,2024-05-04,SKU_1,WH_1,45
181,2024-06-30,SKU_1,WH_1,8
194,2024-07-13,SKU_1,WH_1,33


Por último, filas que no rompen ninguna regla columna a columna pero no cuadran **en conjunto**: la
distancia de Mahalanobis de la lección 03, sobre ventas, pronóstico e inventario.

In [13]:
DIMS = ["Units_Sold", "Demand_Forecast", "Inventory_Level"]
THRESHOLD = np.sqrt(chi2.ppf(0.999, df=len(DIMS)))  # 1 de cada 1.000 filas normales lo supera


def mahalanobis(df, cols):
    X = df[cols].to_numpy(dtype=float)
    diff = X - X.mean(axis=0)
    inv = np.linalg.inv(np.cov(X, rowvar=False))
    d2 = np.einsum("ij,jk,ik->i", diff, inv, diff)
    return pd.Series(np.sqrt(d2), index=df.index, name="distance")


def flag_multi_outliers(df):
    out = df.copy()
    out["mahalanobis"] = mahalanobis(out, DIMS)
    out["multi_outlier"] = out["mahalanobis"] > THRESHOLD
    return out


multi = flag_multi_outliers(raw)
print(f"umbral: distancia > {THRESHOLD:.2f}  ->  {multi['multi_outlier'].sum()} filas marcadas")
multi.loc[multi["multi_outlier"], DIMS + ["mahalanobis"]].sort_values(
    "mahalanobis", ascending=False
).head()

umbral: distancia > 4.03  ->  48 filas marcadas


,Units_Sold,Demand_Forecast,Inventory_Level,mahalanobis
31120,35,49.08,668,5.25
75366,17,29.64,804,4.89
49642,25,15.67,927,4.70
78927,53,61.42,345,4.68
85133,39,50.96,592,4.65


In [14]:
px.scatter_matrix(
    multi.sample(3_000, random_state=SEED),
    dimensions=DIMS,
    color="multi_outlier",
    opacity=0.4,
    title="Ventas, pronóstico e inventario (muestra de 3.000 filas)",
).show()

> **Lectura.** Ninguna fila es imposible. Los días raros para su SKU y su mes, y las filas que se
> salen del conjunto (sobre todo donde el pronóstico y las ventas no coinciden), son valores
> plausibles: **se marcan, no se borran**, y quedan disponibles para revisarlos.

### 2.3 Scorecard de calidad

La columna del *antes* de la lección 03, por dimensión de calidad.

In [15]:
scorecard = pd.DataFrame(
    [
        ("Completitud", "% de celdas nulas", f"{raw.isna().mean().mean():.1%}"),
        ("Unicidad", "claves fecha-SKU-almacén repetidas", f"{n_dup_keys}"),
        ("Validez", "filas imposibles por regla de dominio", f"{impossible.sum()}"),
        ("Validez", "incumplimientos del contrato", f"{len(violations)}"),
        ("Consistencia", "atributos de la combinación que cambian", "Region (4 valores)"),
        ("Utilidad", "columnas constantes", ", ".join(constant)),
        ("Cobertura temporal", "días con datos", f"{n_days} (01/01 a 30/12/2024)"),
    ],
    columns=["dimensión", "medida", "antes"],
)
scorecard

,dimensión,medida,antes
0,Completitud,% de celdas nulas,0.0%
1,Unicidad,claves fecha-SKU-almacén repetidas,0
2,Validez,filas imposibles por regla de dominio,0
3,Validez,incumplimientos del contrato,0
4,Consistencia,atributos de la combinación que cambian,Region (4 valores)
5,Utilidad,columnas constantes,Stockout_Flag
6,Cobertura temporal,días con datos,365 (01/01 a 30/12/2024)


### 2.4 El pipeline: tipos, outliers, cobertura y reglas

Mismo patrón que la lección 03: cada paso devuelve una tabla **nueva** (`out = df.copy()`),
**marca sin borrar** y queda en el log.

La **cobertura** usa solo ventas **anteriores** al día evaluado, de la misma combinación, con al
menos 7 días de historia. Con las herramientas de la lección 04 (`rolling` dentro de
`groupby().transform`): la suma de una ventana de 29 días menos el propio día, entre el número de
días anteriores. El primer día evaluado de cada combinación es el octavo.

In [16]:
def fix_types(df):
    out = df.copy()
    out["Promotion_Flag"] = out["Promotion_Flag"].astype(bool)
    out["Stockout_Flag"] = out["Stockout_Flag"].astype(bool)
    return out


def add_coverage(df):
    out = df.sort_values(KEY).copy()
    out["dias"] = 1
    by_combo = out.groupby(COMBO, observed=True)
    # ventana de 29 días que termina hoy: al restar el propio día quedan los 28 anteriores
    suma = by_combo["Units_Sold"].transform(
        lambda s: s.rolling(WINDOW + 1, min_periods=MIN_DAYS + 1).sum()
    )
    dias = by_combo["dias"].transform(
        lambda s: s.rolling(WINDOW + 1, min_periods=MIN_DAYS + 1).sum()
    )
    out["demanda_media"] = (suma - out["Units_Sold"]) / (dias - 1)
    out["cobertura_calculable"] = out["demanda_media"] > 0  # NaN o 0 -> False
    out["cobertura"] = np.where(
        out["cobertura_calculable"], out["Inventory_Level"] / out["demanda_media"], np.nan
    )
    return out.drop(columns="dias")


def flag_rules(df):
    out = df.copy()
    lead = out["Supplier_Lead_Time_Days"]
    out["riesgo"] = out["cobertura"] < lead  # cobertura NaN -> False
    out["bajo_rp"] = out["Inventory_Level"] < out["Reorder_Point"]
    out["sobrecubierto"] = out["cobertura"] > lead + MARGIN
    out["en_banda"] = out["cobertura_calculable"] & ~out["riesgo"] & ~out["sobrecubierto"]
    return out


STEPS = [fix_types, flag_sales_outliers, flag_multi_outliers, add_coverage, flag_rules]
clean = raw
entries = [{"step": "raw", "rows": len(raw)}]
for step in STEPS:
    clean = clean.pipe(step)
    entries.append({"step": step.__name__, "rows": len(clean)})
log = pd.DataFrame(entries)

assert not clean.duplicated(subset=KEY).any(), "el pipeline rompió la clave"
assert len(clean) == len(raw), "el pipeline borró filas: debe marcar, no borrar"
log

,step,rows
0,raw,91250
1,fix_types,91250
2,flag_sales_outliers,91250
3,flag_multi_outliers,91250
4,add_coverage,91250
5,flag_rules,91250


In [17]:
evaluated = clean[clean["cobertura_calculable"]]
first_evaluated = evaluated.groupby(COMBO, observed=True)["Date"].min()

print(f"registros evaluados: {len(evaluated):,}")
print(f"excluidos: {len(clean) - len(evaluated):,}  ->  sin historia: "
      f"{clean['demanda_media'].isna().sum():,}, demanda media 0: {(clean['demanda_media'] == 0).sum():,}")
print("primer día evaluado de cada combinación:",
      [d.strftime("%d/%m/%Y") for d in first_evaluated.unique()])

registros evaluados: 89,500
excluidos: 1,750  ->  sin historia: 1,750, demanda media 0: 0
primer día evaluado de cada combinación: ['08/01/2024']


> **Gobierno del dato.** El dataset no contiene datos personales, así que no hace falta
> pseudonimizar (lección 03). La licencia es CC0: se pueden publicar los resultados.

## 3. Modelo de datos (lección 02)

La arquitectura del brief pide una **tabla de hechos** y **dimensiones**. Las columnas que no
cambian dentro de una combinación (sección 2) van a la dimensión; lo que cambia cada día, a los
hechos.

In [18]:
dim_combo = clean[
    COMBO + ["Supplier_ID", "Supplier_Lead_Time_Days", "Reorder_Point", "Unit_Cost", "Unit_Price"]
].drop_duplicates()
dim_fecha = pd.DataFrame({"Date": clean["Date"].drop_duplicates()})
dim_fecha["mes"] = dim_fecha["Date"].dt.strftime("%m")
dim_fecha["dia_semana"] = dim_fecha["Date"].dt.strftime("%u")  # 1 = lunes
fact = clean[
    KEY + ["Region", "Units_Sold", "Inventory_Level", "Order_Quantity", "Promotion_Flag",
           "Demand_Forecast", "ventas_outlier", "multi_outlier", "demanda_media",
           "cobertura_calculable", "cobertura", "riesgo", "bajo_rp", "sobrecubierto", "en_banda"]
]

assert len(dim_combo) == n_combos, "un atributo fijo cambia dentro de una combinación"

# reconstruir la tabla de análisis: muchos días por combinación -> m:1
rebuilt = fact.merge(dim_combo, on=COMBO, how="left", validate="m:1").merge(
    dim_fecha, on="Date", how="left", validate="m:1"
)
assert len(rebuilt) == len(fact), "un merge cambió el número de filas"
print(f"hechos: {len(fact):,} filas, dim_combo: {len(dim_combo)}, dim_fecha: {len(dim_fecha)}")
print("reconstrucción sin perder filas ✔")

hechos: 91,250 filas, dim_combo: 250, dim_fecha: 365
reconstrucción sin perder filas ✔


In [19]:
tables = {"fact_inventario": fact, "dim_combo": dim_combo, "dim_fecha": dim_fecha}
for name, table in tables.items():
    table.to_parquet(PROCESSED / f"{name}.parquet", index=False)

manifest = {
    "dataset": "inventario_sago",
    "source": "https://www.kaggle.com/datasets/ziya07/high-dimensional-supply-chain-inventory-dataset",
    "license": "CC0 1.0",
    "downloaded_on": "2026-10-13",
    "processed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "raw_sha256": hashlib.sha256(RAW.read_bytes()).hexdigest()[:16],
    "contract_passed": not violations,
    "tables": {name: {"rows": int(len(t)), "columns": int(t.shape[1])} for name, t in tables.items()},
    "steps": log.to_dict("records"),
}
(PROCESSED / "inventario_sago.manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

sizes = pd.DataFrame(
    {
        "MB": [RAW.stat().st_size / 1024**2]
        + [(PROCESSED / f"{n}.parquet").stat().st_size / 1024**2 for n in tables]
    },
    index=["CSV original"] + [f"{n}.parquet" for n in tables],
).round(2)
sizes

{
  "dataset": "inventario_sago",
  "source": "https://www.kaggle.com/datasets/ziya07/high-dimensional-supply-chain-inventory-dataset",
  "license": "CC0 1.0",
  "downloaded_on": "2026-10-13",
  "processed_at": "2026-10-17T20:52:16+00:00",
  "raw_sha256": "76dad280541a60f1",
  "contract_passed": true,
  "tables": {
    "fact_inventario": {
      "rows": 91250,
      "columns": 18
    },
    "dim_combo": {
      "rows": 250,
      "columns": 7
    },
    "dim_fecha": {
      "rows": 365,
      "columns": 3
    }
  },
  "steps": [
    {
      "step": "raw",
      "rows": 91250
    },
    {
      "step": "fix_types",
      "rows": 91250
    },
    {
      "step": "flag_sales_outliers",
      "rows": 91250
    },
    {
      "step": "flag_multi_outliers",
      "rows": 91250
    },
    {
      "step": "add_coverage",
      "rows": 91250
    },
    {
      "step": "flag_rules",
      "rows": 91250
    }
  ]
}


,MB
CSV original,6.25
fact_inventario.parquet,1.28
dim_combo.parquet,0.01
dim_fecha.parquet,0.01


## 4. Exploración (lección 04)

### 4.1 Primer vistazo

In [20]:
clean[
    ["Units_Sold", "Inventory_Level", "Supplier_Lead_Time_Days", "Reorder_Point",
     "Unit_Cost", "Unit_Price", "cobertura"]
].describe()

,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Unit_Cost,Unit_Price,cobertura
count,"91,250.00","91,250.00","91,250.00","91,250.00","91,250.00","91,250.00","89,500.00"
mean,20.05,471.52,7.98,300.07,12.20,18.26,27.32
std,9.07,133.49,3.91,54.88,4.57,7.12,14.31
min,0.00,168.00,2.00,201.00,5.02,6.95,5.54
25%,13.00,370.00,4.00,252.00,8.18,12.00,16.54
50%,20.00,461.00,8.00,300.00,11.99,18.18,23.31
75%,27.00,564.00,11.00,346.00,16.32,23.39,35.23
max,59.00,990.00,14.00,398.00,19.76,35.10,100.96


In [21]:
clean.describe(include=["category", "bool"])

,SKU_ID,Warehouse_ID,Supplier_ID,Region,Promotion_Flag,Stockout_Flag,ventas_outlier,multi_outlier,cobertura_calculable,riesgo,bajo_rp,sobrecubierto,en_banda
count,91250,91250,91250,91250,91250,91250,91250,91250,91250,91250,91250,91250,91250
unique,50,5,10,4,2,1,2,2,2,2,2,2,2
top,SKU_1,WH_1,SUP_7,North,False,False,False,False,True,False,False,True,False
freq,1825,18250,12410,22974,81980,91250,90094,91202,89500,87951,86463,49837,54886


| Tipo de columna | Ejemplos | Se resume con | Se dibuja con |
|---|---|---|---|
| Numérica continua | `cobertura`, `Unit_Cost` | media, mediana, desviación, IQR | histograma, boxplot |
| Numérica de conteo | `Units_Sold`, `Inventory_Level` | igual, vigilando la asimetría | histograma, boxplot |
| Categórica | `Warehouse_ID`, `Supplier_ID` | conteos, porcentajes, moda | barras |
| Booleana | `Promotion_Flag`, `riesgo` | % de `True` | barras o un número |
| Tiempo | `Date` | primero, último, frecuencia | línea |

### 4.2 ¿Qué es un valor típico?

In [22]:
u = clean["Units_Sold"]
print(f"media   {u.mean():8.1f}")
print(f"mediana {u.median():8.1f}")
print(f"moda    {u.mode().iloc[0]:8.0f}")
print(f"skew    {u.skew():8.2f}   (0 = simétrica)")

fig = px.histogram(clean, x="Units_Sold", nbins=60, title="Unidades vendidas por día y combinación")
fig.add_vline(x=u.mean(), line_dash="dash", line_color="firebrick",
              annotation_text=f"media {u.mean():.1f}", annotation_position="top right")
fig.add_vline(x=u.median(), line_dash="dot", line_color="black",
              annotation_text=f"mediana {u.median():.0f}", annotation_position="top left")
fig.update_layout(bargap=0.05)
fig.show()

media       20.1
mediana     20.0
moda          20
skew        0.14   (0 = simétrica)


In [23]:
def trimmed_mean(s, share=0.05):
    lo, hi = s.quantile([share, 1 - share])
    return s[(s >= lo) & (s <= hi)].mean()


by_wh = clean.groupby("Warehouse_ID", observed=True)["Units_Sold"]
centre = pd.DataFrame(
    {
        "media": by_wh.mean(),
        "media recortada (5%)": by_wh.apply(trimmed_mean),
        "mediana": by_wh.median(),
        "moda": by_wh.agg(lambda s: s.mode().iloc[0]),
    }
)
centre

,media,media recortada (5%),mediana,moda
Warehouse_ID,,,,
WH_1,20.10,20.08,20.00,23
WH_2,20.06,20.03,20.00,20
WH_3,20.06,20.05,20.00,19
WH_4,20.04,20.01,20.00,24
WH_5,20.01,19.98,20.00,16


> **Lectura.** Media y mediana casi iguales y asimetría cercana a 0: a diferencia de la mayoría de
> datos de negocio, las ventas diarias son **simétricas**. Los cinco almacenes son prácticamente
> iguales.

### 4.3 ¿Cuánto varía?

In [24]:
spread = clean.groupby("Warehouse_ID", observed=True)["Units_Sold"].agg(
    mean="mean",
    std="std",
    p10=lambda s: s.quantile(0.10),
    q1=lambda s: s.quantile(0.25),
    median="median",
    q3=lambda s: s.quantile(0.75),
    p90=lambda s: s.quantile(0.90),
)
spread["iqr"] = spread["q3"] - spread["q1"]
spread["cv"] = spread["std"] / spread["mean"]
spread.sort_values("cv", ascending=False)

,mean,std,p10,q1,median,q3,p90,iqr,cv
Warehouse_ID,,,,,,,,,
WH_3,20.06,9.15,8.00,13.00,20.00,27.00,32.00,14.00,0.46
WH_5,20.01,9.05,8.00,13.00,20.00,27.00,32.00,14.00,0.45
WH_4,20.04,9.07,8.00,13.00,20.00,27.00,32.00,14.00,0.45
WH_2,20.06,9.07,8.00,13.00,20.00,27.00,32.00,14.00,0.45
WH_1,20.10,9.00,8.00,13.00,20.00,27.00,32.00,14.00,0.45


### 4.4 ¿Está concentrada la demanda? (la H1 original)

La primera versión del brief suponía que pocas combinaciones concentraban las ventas.

In [25]:
by_combo = (
    clean.groupby(COMBO, observed=True)["Units_Sold"].sum().sort_values(ascending=False)
)
top20 = by_combo.head(len(by_combo) // 5).sum() / by_combo.sum()
cv_combo = by_combo.std() / by_combo.mean()
print(f"el 20 % de las combinaciones con más ventas vende el {top20:.1%} de las unidades")
print(f"coeficiente de variación de las ventas anuales entre combinaciones: {cv_combo:.1%}")

el 20 % de las combinaciones con más ventas vende el 20.3% de las unidades
coeficiente de variación de las ventas anuales entre combinaciones: 1.3%


> **Lectura.** El 20 % vende el 20 %: la demanda está repartida por igual. La hipótesis de
> concentración era falsa, y por eso el brief la sustituyó por la de estacionalidad.

### 4.5 Tendencias en el tiempo

In [26]:
daily = clean.groupby("Date", as_index=False).agg(
    units=("Units_Sold", "sum"), inventory=("Inventory_Level", "sum")
)
daily["media 28 días"] = daily["units"].rolling(WINDOW).mean()

fig = px.line(daily, x="Date", y=["units", "media 28 días"],
              title="Unidades vendidas al día, todas las combinaciones")
fig.update_layout(yaxis_title="unidades", legend_title=None)
fig.show()

In [27]:
month = clean["Date"].dt.strftime("%m")
monthly = clean.groupby(month).agg(
    demanda=("Units_Sold", "mean"), inventario=("Inventory_Level", "mean")
)
monthly.index.name = "mes"
peak_valley = monthly["demanda"].max() / monthly["demanda"].min()

fig = px.bar(monthly.reset_index(), x="mes", y="demanda", text_auto=".1f",
             title=f"Demanda media por mes: el pico es {peak_valley:.1f} veces el valle")
fig.update_layout(yaxis_title="unidades por día y combinación")
fig.show()
monthly

,demanda,inventario
mes,,
01,22.80,527.47
02,27.20,459.95
03,29.88,459.07
04,29.84,454.42
05,27.21,459.08
06,22.67,460.49
07,17.41,466.50
08,12.78,477.49
09,10.17,472.32


> **Lectura.** La demanda se triplica entre septiembre y marzo, mientras el inventario medio apenas
> se mueve. Es la base de la H1. ¿Comparten todos los SKU la misma curva? ¿Hay un patrón semanal?

In [28]:
sku_month = clean.groupby([clean["SKU_ID"], month], observed=True)["Units_Sold"].mean().unstack()
seasonal_corr = sku_month.T.corr()
print(f"correlación mínima entre las curvas mensuales de dos SKU: {seasonal_corr.min().min():.3f}")

DAYS = {"1": "lun", "2": "mar", "3": "mié", "4": "jue", "5": "vie", "6": "sáb", "7": "dom"}
weekday = clean.groupby(clean["Date"].dt.strftime("%u"))["Units_Sold"].mean()
weekday.index = weekday.index.map(DAYS)
pd.DataFrame({"venta media": weekday}).T

correlación mínima entre las curvas mensuales de dos SKU: 0.990


Date,lun,mar,mié,jue,vie,sáb,dom
venta media,20.08,20.06,19.99,20.04,20.08,20.02,20.11


> **Lectura.** Todas las curvas son prácticamente iguales (datos generados) y no hay patrón
> semanal: por eso se descartó unir el calendario de festivos.

### 4.6 Correlación

In [29]:
cols = ["Units_Sold", "Inventory_Level", "Supplier_Lead_Time_Days", "Reorder_Point",
        "Unit_Cost", "Unit_Price", "Demand_Forecast"]
pearson = clean[cols].corr()
spearman = clean[cols].corr(method="spearman")

fig = px.imshow(pearson, text_auto=".2f", zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r", title="Correlación de Pearson (filas diarias)")
fig.show()
(spearman - pearson).round(2)

,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Unit_Cost,Unit_Price,Demand_Forecast
Units_Sold,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00
Inventory_Level,0.00,0.00,-0.00,-0.01,-0.00,0.00,0.00
Supplier_Lead_Time_Days,0.00,-0.00,0.00,-0.00,-0.00,0.00,0.00
Reorder_Point,0.00,-0.01,-0.00,0.00,-0.00,0.01,0.00
Unit_Cost,-0.00,-0.00,-0.00,-0.00,0.00,0.01,-0.00
Unit_Price,-0.00,0.00,0.00,0.01,0.01,0.00,-0.00
Demand_Forecast,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00


A nivel diario casi todo es ruido. Los atributos fijos se comparan mejor **por combinación**:

In [30]:
per_combo = clean.groupby(COMBO, observed=True).agg(
    plazo=("Supplier_Lead_Time_Days", "first"),
    punto_pedido=("Reorder_Point", "first"),
    inventario_medio=("Inventory_Level", "mean"),
    demanda_media=("Units_Sold", "mean"),
    coste=("Unit_Cost", "first"),
)
combo_corr = per_combo.corr()
fig = px.imshow(combo_corr, text_auto=".2f", zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r", title="Correlación entre combinaciones (250)")
fig.show()

> **Lectura.** El punto de pedido sigue al **inventario medio** (r = 0,98), pero no al plazo de
> suministro ni a la demanda (r cercano a 0). Una regla de reposición que no mira el plazo es justo lo que
> pone a prueba la H2. Y el inventario tampoco responde a la demanda: de ahí sale la H1.

## 5. Hipótesis

Todos los porcentajes usan como denominador los registros **evaluados** (con cobertura
calculable).

### H1. El riesgo se concentra en los meses de demanda alta

In [31]:
m = evaluated["Date"].dt.strftime("%m")
by_month = evaluated.groupby(m)[["riesgo", "sobrecubierto", "bajo_rp"]].mean()
by_month.index.name = "mes"
h1_share = evaluated.loc[m.isin(["02", "03", "04", "05"]), "riesgo"].sum() / evaluated["riesgo"].sum()

fig = px.line(by_month.reset_index(), x="mes", y=["riesgo", "sobrecubierto"],
              title=f"Riesgo y sobrecobertura por mes: el {h1_share:.1%} del riesgo cae entre febrero y mayo")
fig.update_layout(yaxis_title="% de registros evaluados", yaxis_tickformat=".0%", legend_title=None)
fig.show()
by_month.style.format("{:.1%}")

,riesgo,sobrecubierto,bajo_rp
mes,,,
01,1.6%,51.4%,5.1%
02,5.3%,28.9%,7.5%
03,9.6%,16.9%,8.1%
04,11.1%,12.3%,8.3%
05,9.8%,17.5%,7.5%
06,5.1%,29.2%,6.4%
07,1.0%,56.0%,4.7%
08,0.0%,83.8%,3.2%
09,0.0%,96.5%,2.6%


In [32]:
risk_map = evaluated.assign(mes=m).pivot_table(
    index="Warehouse_ID", columns="mes", values="riesgo", aggfunc="mean", observed=True
)
fig = px.imshow(risk_map, text_auto=".0%", color_continuous_scale="Reds",
                title="% de registros en riesgo, por almacén y mes")
fig.show()

> **Lectura.** El mismo patrón en los cinco almacenes: el total no esconde ningún almacén con un
> problema propio.

### H2. ¿Priorizan casos distintos las dos reglas?

La tabla cruzada de la lección 03: qué marca cada regla y dónde coinciden.

In [33]:
cross = pd.crosstab(
    evaluated["riesgo"], evaluated["bajo_rp"],
    rownames=["cobertura < plazo"], colnames=["Inventory_Level < Reorder_Point"],
)
h2_missed = (evaluated["riesgo"] & ~evaluated["bajo_rp"]).sum() / evaluated["riesgo"].sum()
print(f"del riesgo por cobertura, la regla del punto de pedido no marca el {h2_missed:.1%}")
cross

del riesgo por cobertura, la regla del punto de pedido no marca el 69.3%


Inventory_Level < Reorder_Point,False,True
cobertura < plazo,,
False,82431,3770
True,2286,1013


In [34]:
combo_risk = evaluated.groupby(COMBO, observed=True).agg(
    plazo=("Supplier_Lead_Time_Days", "first"), riesgo=("riesgo", "mean")
)
combo_risk["plazo >= 8 días"] = combo_risk["plazo"] >= 8

fig = px.box(combo_risk.reset_index(), x="plazo", y="riesgo", points="all",
             title="Riesgo por combinación según el plazo de suministro")
fig.update_layout(yaxis_title="% de días en riesgo", yaxis_tickformat=".0%")
fig.show()
combo_risk.groupby("plazo >= 8 días")["riesgo"].mean().map("{:.2%}".format)

plazo >= 8 días
False    0.01%
True     6.92%
Name: riesgo, dtype: str

In [35]:
supplier = evaluated.assign(valor=evaluated["Inventory_Level"] * evaluated["Unit_Cost"]).groupby(
    "Supplier_ID", observed=True
).agg(
    plazo_medio=("Supplier_Lead_Time_Days", "mean"),
    en_riesgo=("riesgo", "mean"),
    bajo_rp=("bajo_rp", "mean"),
    valor=("valor", "sum"),
).reset_index()

fig = px.scatter(
    supplier,
    x="plazo_medio",
    y="en_riesgo",
    size="valor",
    text="Supplier_ID",
    size_max=40,
    title="Proveedores: plazo medio vs % de registros en riesgo (burbuja = valor del inventario)",
)
fig.update_traces(textposition="top center")
fig.update_layout(yaxis_tickformat=".1%")
fig.show()

> **Lectura.** Las dos reglas no son equivalentes, y el riesgo por cobertura aparece casi solo en
> plazos largos. Sin roturas reales (`Stockout_Flag` siempre 0) no se puede decir cuál de las dos
> acierta más: es una comparación de reglas, no una demostración de que una falla.

### H3. ¿Venden más las promociones?

Primero el *lift* de la lección 04, por mes.

In [36]:
lift = (
    evaluated.groupby([m, "Promotion_Flag"])["Units_Sold"]
    .mean()
    .unstack("Promotion_Flag")
)
lift.columns = ["sin_promo", "con_promo"]  # unstack ordena False, True
lift.index.name = "mes"
lift["lift"] = lift["con_promo"] / lift["sin_promo"] - 1

fig = px.bar(lift.reset_index(), x="mes", y="lift", text_auto="+.0%",
             title="Lift de las promociones por mes")
fig.update_layout(yaxis_tickformat="+.0%")
fig.show()

La comparación justa de la lección 04 mantiene fijo el producto: **el mismo SKU, en el mismo
almacén**, con y sin promoción. Para que la estacionalidad no entre, cada par usa solo los meses en
los que esa combinación tiene días de los dos tipos.

In [37]:
g = (
    evaluated.groupby(COMBO + [m, "Promotion_Flag"], observed=True)["Units_Sold"]
    .mean()
    .unstack("Promotion_Flag")
)
g.columns = ["sin_promo", "con_promo"]
months_without_both = int(g.isna().any(axis=1).sum())
strict = g.dropna()  # un par por SKU-almacén y mes
pairs = strict.reset_index().groupby(COMBO, observed=True)[["sin_promo", "con_promo"]].mean()
uplift = pairs["con_promo"] / pairs["sin_promo"] - 1  # un par por SKU-almacén
uplift_strict = strict["con_promo"] / strict["sin_promo"] - 1

h3 = pd.DataFrame(
    {
        "pares": [len(pairs), len(strict)],
        "uplift mediano": [uplift.median(), uplift_strict.median()],
        "% de pares con más ventas en promoción": [(uplift > 0).mean(), (uplift_strict > 0).mean()],
    },
    index=["por SKU-almacén (principal)", "por SKU-almacén y mes (robustez)"],
)
print(f"combinación-mes sin días de los dos tipos (excluidas): {months_without_both}")
h3.style.format({"uplift mediano": "{:+.1%}", "% de pares con más ventas en promoción": "{:.0%}"})

combinación-mes sin días de los dos tipos (excluidas): 151


,pares,uplift mediano,% de pares con más ventas en promoción
por SKU-almacén (principal),250,+28.1%,100%
por SKU-almacén y mes (robustez),2849,+28.2%,87%


In [38]:
fig = px.histogram(uplift.rename("uplift").reset_index(), x="uplift", nbins=30,
                   title="Uplift por SKU-almacén: ninguna combinación vende menos en promoción")
fig.add_vline(x=0, line_dash="dot", line_color="black")
fig.update_layout(xaxis_tickformat="+.0%")
fig.show()

## 6. KPIs: valores actuales

Como la tabla de KPIs por tienda de la lección 04: tasas, no conteos, para poder comparar almacenes
de distinto tamaño. Son porcentajes de **registros diarios**; el dashboard los agregará por semana.

In [39]:
evaluated = evaluated.assign(valor=evaluated["Inventory_Level"] * evaluated["Unit_Cost"])
kpi = evaluated.groupby("Warehouse_ID", observed=True).agg(
    demanda_media=("Units_Sold", "mean"),
    en_riesgo=("riesgo", "mean"),
    en_banda=("en_banda", "mean"),
    bajo_rp=("bajo_rp", "mean"),
    sobrecubierto=("sobrecubierto", "mean"),
    valor=("valor", "sum"),
)
kpi["valor_diario_medio"] = kpi["valor"] / evaluated["Date"].nunique()
kpi.index = kpi.index.astype(str)
kpi.loc["TOTAL"] = [
    evaluated["Units_Sold"].mean(),
    evaluated["riesgo"].mean(),
    evaluated["en_banda"].mean(),
    evaluated["bajo_rp"].mean(),
    evaluated["sobrecubierto"].mean(),
    evaluated["valor"].sum(),
    evaluated["valor"].sum() / evaluated["Date"].nunique(),
]
kpi = kpi.drop(columns="valor")
kpi.style.format(
    {
        "demanda_media": "{:.1f}",
        "en_riesgo": "{:.1%}",
        "en_banda": "{:.1%}",
        "bajo_rp": "{:.1%}",
        "sobrecubierto": "{:.1%}",
        "valor_diario_medio": "{:,.0f}",  # moneda sin verificar
    }
)

,demanda_media,en_riesgo,en_banda,bajo_rp,sobrecubierto,valor_diario_medio
Warehouse_ID,,,,,,
WH_1,20.1,3.8%,42.0%,5.3%,54.3%,"267,831"
WH_2,20.0,3.1%,39.8%,5.3%,57.1%,"330,060"
WH_3,20.0,4.5%,43.4%,5.4%,52.1%,"263,726"
WH_4,20.0,4.5%,40.1%,5.3%,55.5%,"275,048"
WH_5,20.0,2.5%,38.0%,5.4%,59.5%,"290,927"
TOTAL,20.0,3.7%,40.6%,5.3%,55.7%,"1,427,592"


## 7. Cifras del brief

Cada cifra del `project-brief.md`, comprobada. Si el CSV cambia y una cifra deja de ser cierta,
esta celda falla y hay que actualizar el brief.

In [40]:
# --- checks del brief (no editar sin actualizar el brief) ---
over_aug_dec = by_month.loc["08":"12", "sobrecubierto"]

BRIEF = pd.DataFrame(
    [
        ("filas x columnas", f"{raw.shape[0]:,} x {raw.shape[1]}", "91,250 x 15"),
        ("SKU x almacenes", f"{raw['SKU_ID'].nunique()} x {raw['Warehouse_ID'].nunique()}", "50 x 5"),
        ("registros evaluados", f"{len(evaluated):,}", "89,500"),
        ("% en riesgo", f"{evaluated['riesgo'].mean():.1%}", "3.7%"),
        ("% bajo punto de pedido", f"{evaluated['bajo_rp'].mean():.1%}", "5.3%"),
        ("% en banda", f"{evaluated['en_banda'].mean():.0%}", "41%"),
        ("H1: riesgo feb-may", f"{h1_share:.0%}", "83%"),
        ("H2: riesgo no marcado por RP", f"{h2_missed:.0%}", "69%"),
        ("H3: uplift mediano por SKU-almacén", f"{uplift.median():.0%}", "28%"),
        ("H3: pares válidos", f"{len(pairs)}", "250"),
        ("H3: uplift mediano por SKU-almacén-mes", f"{uplift_strict.median():.0%}", "28%"),
        ("sobrecubierto ago-dic (mín.)", f"{over_aug_dec.min():.0%}", "81%"),
        ("Demand_Forecast: corr. mismo día", f"{leak.iloc[0, 0]:.2f}", "0.95"),
        ("Demand_Forecast: corr. día adyacente", f"{leak.iloc[1, 0]:.2f}", "0.61"),
        ("corr. mínima entre curvas de SKU", f"{seasonal_corr.min().min():.2f}", "0.99"),
        ("filas imposibles", f"{impossible.sum()}", "0"),
        ("días raros para su SKU y su mes", f"{clean['ventas_outlier'].mean():.1%}", "1.3%"),
        ("filas marcadas por Mahalanobis", f"{clean['multi_outlier'].sum()}", "48"),
        ("corr. plazo y punto de pedido (combinaciones)", f"{combo_corr.loc['plazo', 'punto_pedido']:.2f}", "0.03"),
    ],
    columns=["cifra", "calculado", "en el brief"],
)
BRIEF["ok"] = BRIEF["calculado"] == BRIEF["en el brief"]
assert BRIEF["ok"].all(), BRIEF.loc[~BRIEF["ok"]]
print("✅ todas las cifras del brief coinciden con los datos")
BRIEF

✅ todas las cifras del brief coinciden con los datos


,cifra,calculado,en el brief,ok
0,filas x columnas,"91,250 x 15","91,250 x 15",True
1,SKU x almacenes,50 x 5,50 x 5,True
2,registros evaluados,"89,500","89,500",True
3,% en riesgo,3.7%,3.7%,True
4,% bajo punto de pedido,5.3%,5.3%,True
5,% en banda,41%,41%,True
6,H1: riesgo feb-may,83%,83%,True
7,H2: riesgo no marcado por RP,69%,69%,True
8,H3: uplift mediano por SKU-almacén,28%,28%,True
9,H3: pares válidos,250,250,True
